In [1]:
import time
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Load data
try:
    df = pd.read_parquet("data/analysis_sample.parquet")
except Exception:
    df = pd.read_csv(r"C:\Users\ugochukwu.emenike\PycharmProjects\PythonProject\powerbi\analysis_sample.csv")

# 2. Target & strict leakage isolation
df = df.dropna(subset=['SYSTOLIC_BP', 'DIASTOLIC_BP']).copy()
df['HIGH_BP_FLAG'] = np.where((df['SYSTOLIC_BP'] >= 140) | (df['DIASTOLIC_BP'] >= 90), 1, 0)

leakage_cols = ['SYSTOLIC_BP', 'DIASTOLIC_BP', 'DIAGNOSED_HYPERTENSION', 'BP_CATEGORY', 'SAMPLE_ID', 'HIGH_BP_FLAG']
X = df.drop(columns=[col for col in leakage_cols if col in df.columns])
y = df['HIGH_BP_FLAG']

# 3. Stratified split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 4. Preprocessing pipelines
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'category', 'string']).columns.tolist()

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, numeric_features),
        ('cat', cat_pipeline, categorical_features)
    ]
)

# 5. Define all four models
models = {
    "1. Dummy Baseline": DummyClassifier(strategy='most_frequent'),
    "2. Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "3. Decision Tree (depth=5)": DecisionTreeClassifier(max_depth=5, random_state=42),
    "4. Random Forest (100 trees)": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=42)
}

# 6. Fit and benchmark each model
results = []

for name, clf in models.items():
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', clf)
    ])

    start_time = time.time()
    pipe.fit(X_train, y_train)
    fit_time = time.time() - start_time

    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1] if hasattr(clf, "predict_proba") else [0] * len(y_test)

    # Dummy classifier predict_proba gives constant probabilities; default roc_auc to 0.50
    try:
        auc = roc_auc_score(y_test, y_proba)
    except ValueError:
        auc = 0.5000

    results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "ROC-AUC": round(auc, 4),
        "Precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "Recall": round(recall_score(y_test, y_pred, zero_division=0), 4),
        "Fit Time (s)": round(fit_time, 4)
    })

# 7. Display comparison table
comparison_df = pd.DataFrame(results)
print("=== Model Comparison Table (US-10 Triage) ===")
print(comparison_df.to_string(index=False))

=== Model Comparison Table (US-10 Triage) ===
                       Model  Accuracy  ROC-AUC  Precision  Recall  Fit Time (s)
           1. Dummy Baseline    0.5941   0.5000     0.5941  1.0000        0.3597
      2. Logistic Regression    0.5933   0.6119     0.6138  0.8507        0.4786
  3. Decision Tree (depth=5)    0.5916   0.6111     0.6017  0.9244        0.6655
4. Random Forest (100 trees)    0.5825   0.5980     0.6326  0.7092        3.3102


### Model Trade-off Evaluation: Is the Complexity Worth It?

**Answer for Chidi:**
While Random Forest provides marginal metric gains over Logistic Regression, the small lift in ROC-AUC does not justify the loss of interpretability and added deployment complexity for community clinic triage, making **Logistic Regression (or a shallow Decision Tree)** the preferred choice where rapid nurse buy-in, direct risk-weight explainability, and transparent odds ratios are critical.